# Transformer Architecture from Scratch

In [41]:
import torch 
import torch.nn as nn

In [42]:
class SelfAttention(nn.Module): 
    def __init__(self, embed_size, heads): 
        super(SelfAttention, self).__init__() 
        self.embed_size = embed_size 
        self.heads = heads 
        self.head_dim = embed_size // heads 
        
        assert(self.head_dim * heads == embed_size), "Embed size needs to be div by heads" 
        
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False) 
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False) 
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False) 
        self.fc_out = nn.Linear(heads*self.head_dim, embed_size) 
        
    def forward(self, values, keys, query, mask): 
        N = query.shape[0] 
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1] 
        
        # Split embedding into self.heads pieces 
        values = values.reshape(N, value_len, self.heads, self.head_dim) 
        keys = keys.reshape(N, key_len, self.heads, self.head_dim) 
        queries = query.reshape(N, query_len, self.heads, self.head_dim) 

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)
        
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys]) 
        # queries shape: (N, query_len, heads, heads_dim) 
        # keys shape: (N, key_len, heads, heads_dim) 
        # energy shape: (N, heads, query_len, key_len) 
        
        if mask is not None: 
            energy = energy.masked_fill(mask == 0, float("-1e20")) 
        
        attention = torch.softmax(energy / (self.embed_size ** (1/2)), dim=3) 
        
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads*self.head_dim
        ) 
        
        out = self.fc_out(out)
        return out

## Self-Attention Layer Dimensions

Input dimensions:
- Batch size (N)
- Sequence length (seq_len)
- Embedding dimension (embed_size)
- Number of attention heads (heads)
- Head dimension (head_dim = embed_size // heads)

Transformations:
1. Input shape: (N, seq_len, embed_size)
2. Split into heads: (N, seq_len, heads, head_dim)
3. Query, Key, Value projections: Each maintains shape (N, seq_len, heads, head_dim)
4. Energy (attention scores): (N, heads, query_len, key_len)
5. Attention output: (N, seq_len, embed_size)

The final output maintains the input dimensions (N, seq_len, embed_size)

In [43]:
class TransformerBlock(nn.Module): 
    def __init__(self, embed_size, heads, dropout, forward_expansion): 
        super (TransformerBlock, self).__init__() 
        self.attention = SelfAttention(embed_size, heads) 
        self.norm1 = nn. LayerNorm(embed_size) 
        self.norm2 = nn. LayerNorm(embed_size) 
        
        self.feed_forward = nn.Sequential( 
            nn.Linear(embed_size, forward_expansion*embed_size), 
            nn.ReLU(), 
            nn.Linear(forward_expansion*embed_size, embed_size) 
        ) 
        self.dropout = nn. Dropout(dropout)

    def forward(self, value, key, query, mask): 
        attention = self.attention (value, key, query, mask) 
        
        x = self.dropout(self.norm1(attention + query)) 
        forward = self.feed_forward(x) 
        out = self.dropout(self.norm2(forward + x)) 
        return out

## Transformer Block Dimensions

Input dimensions:
- Same as Self-Attention layer
- Forward expansion factor for feed-forward network

Transformations:
1. Self-Attention: (N, seq_len, embed_size) → (N, seq_len, embed_size)
2. Add & Norm: Maintains shape
3. Feed Forward:
   - First linear: (N, seq_len, embed_size) → (N, seq_len, forward_expansion*embed_size)
   - ReLU activation
   - Second linear: (N, seq_len, forward_expansion*embed_size) → (N, seq_len, embed_size)
4. Add & Norm: Maintains shape

Final output shape: (N, seq_len, embed_size)

In [44]:
class Encoder(nn.Module): 
    def __init__( 
        self, 
        src_vocab_size, 
        embed_size, 
        num_layers, 
        heads, 
        device, 
        forward_expansion, 
        dropout, 
        max_length, 
    ): 
        super(Encoder, self).__init__() 
        self.embed_size = embed_size 
        self.device = device 
        self.word_embedding = nn.Embedding(src_vocab_size, embed_size) 
        self.position_embedding = nn.Embedding(max_length, embed_size) 
        
        self.layers = nn.ModuleList( 
            [ 
                TransformerBlock( 
                    embed_size, 
                    heads, 
                    dropout=dropout, 
                    forward_expansion=forward_expansion, 
                ) 
                for _ in range(num_layers)
            ] 
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask): 
        N, seq_length = x.shape 
        positions = torch.arange(0, seq_length).expand(N, seq_length).to(self.device) 
        
        out = self.dropout(self.word_embedding(x) + self.position_embedding(positions)) 
        
        for layer in self.layers: 
            out = layer(out, out, out, mask) 
            
        return out

## Encoder Dimensions

Input dimensions:
- Source vocabulary size
- Embedding dimension
- Maximum sequence length

Transformations:
1. Input tokens: (N, seq_length)
2. Word Embedding: (N, seq_length) → (N, seq_length, embed_size)
3. Positional Encoding: Added to embeddings, maintains shape
4. Multiple Transformer Blocks: Each maintains (N, seq_length, embed_size)

Final output shape: (N, seq_length, embed_size)

In [45]:
class DecoderBlock(nn.Module): 
    def __init__(self, embed_size, heads, forward_expansion, dropout, device): 
        super (DecoderBlock, self).__init__() 
        self.attention = SelfAttention(embed_size, heads) 
        self.norm = nn. LayerNorm(embed_size) 
        self.transformer_block = TransformerBlock( 
            embed_size, heads, dropout, forward_expansion 
        ) 
        self.dropout = nn. Dropout (dropout) 
    
    def forward(self, x, value, key, src_mask, trg_mask): 
        attention = self.attention(x, x, x, trg_mask) 
        query = self.dropout(self.norm(attention + x)) 
        out = self.transformer_block(value, key, query, src_mask) 
        return out

## Decoder Block Dimensions

Input dimensions:
- Target sequence input
- Encoder output
- Source and target masks

Transformations:
1. Masked Self-Attention: (N, seq_len, embed_size) → (N, seq_len, embed_size)
2. Add & Norm: Maintains shape
3. Cross-Attention with encoder output: (N, seq_len, embed_size) → (N, seq_len, embed_size)
4. Feed Forward and final Add & Norm: Maintains shape

Final output shape: (N, seq_len, embed_size)

In [46]:
class Decoder(nn.Module): 
    def __init__( 
        self, 
        trg_vocab_size, 
        embed_size, 
        num_layers, 
        heads, 
        forward_expansion, 
        dropout, 
        device, 
        max_length, 
    ): 
        super(Decoder, self).__init__() 
        self.device = device 
        self.word_embedding = nn.Embedding(trg_vocab_size, embed_size) 
        self.position_embedding = nn.Embedding(max_length, embed_size)
        
        self.layers = nn.ModuleList( 
            [DecoderBlock(embed_size, heads, forward_expansion, dropout, device) 
            for _ in range(num_layers)] 
        )
        
        self.fc_out = nn.Linear(embed_size, trg_vocab_size) 
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, trg_mask): 
        N, seq_length = x.shape 
        positions = torch.arange(0, seq_length).expand (N, seq_length).to(self.device) 
        x = self.dropout((self.word_embedding(x) + self.position_embedding (positions))) 
        
        for layer in self.layers: 
            x = layer(x, enc_out, enc_out, src_mask, trg_mask) 
        
        out = self.fc_out(x)
        return out

## Full Decoder Dimensions

Input dimensions:
- Target vocabulary size
- Embedding dimension
- Maximum sequence length

Transformations:
1. Input tokens: (N, seq_length)
2. Word & Positional Embeddings: (N, seq_length) → (N, seq_length, embed_size)
3. Multiple Decoder Blocks: Each maintains (N, seq_length, embed_size)
4. Final Linear Layer: (N, seq_length, embed_size) → (N, seq_length, trg_vocab_size)

Final output shape: (N, seq_length, target_vocab_size)

In [47]:
class Transformer (nn.Module): 
    def __init__( 
        self, 
        src_vocab_size, 
        trg_vocab_size, 
        src_pad_idx, 
        trg_pad_idx, 
        embed_size=256, 
        num_layers=6, 
        forward_expansion=4, 
        heads=8, 
        dropout=0, 
        device="cuda", 
        max_length=100 
    ): 
        super (Transformer, self).__init__()
        
        self.encoder = Encoder( 
            src_vocab_size, 
            embed_size, 
            num_layers, 
            heads, 
            device, 
            forward_expansion, 
            dropout, 
            max_length 
        )
        
        self.decoder = Decoder( 
            trg_vocab_size, 
            embed_size, 
            num_layers, 
            heads, 
            forward_expansion, 
            dropout, 
            device, 
            max_length 
        ) 
        
        self.src_pad_idx = src_pad_idx 
        self.trg_pad_idx = trg_pad_idx 
        self.device = device

    def make_src_mask(self, src): 
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze (2) 
        # (N, 1, 1, src_len) 
        return src_mask.to(self.device) 
    
    def make_trg_mask(self, trg): 
        N, trg_len = trg.shape 
        trg_mask = torch.tril(torch.ones((trg_len, trg_len))).expand( 
            N, 1, trg_len, trg_len 
        ) 
        return trg_mask.to(self.device)

    def forward(self, src, trg): 
        src_mask = self.make_src_mask(src) 
        trg_mask = self.make_trg_mask(trg) 
        enc_src = self.encoder(src, src_mask) 
        out = self.decoder(trg, enc_src, src_mask, trg_mask) 
        return out

## Complete Transformer Architecture Dimensions

Input dimensions:
- Source sequence: (N, src_seq_len)
- Target sequence: (N, trg_seq_len)

Key transformations:
1. Encoder: (N, src_seq_len) → (N, src_seq_len, embed_size)
2. Decoder: 
   - Takes encoder output (N, src_seq_len, embed_size)
   - Processes target (N, trg_seq_len)
   - Outputs (N, trg_seq_len, trg_vocab_size)

Final output represents probability distribution over target vocabulary for each position

In [57]:
if __name__ == "__main__": 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
    
    x = torch.tensor ([[1, 5, 6, 4, 3, 9, 5, 2, 0], [1, 8, 7, 3, 4, 5, 6, 7, 2]]).to( 
        device 
    ) 
    
    trg = torch.tensor ([[1, 7, 4, 3, 5, 9, 2, 0], [1, 5, 6, 2, 4, 7, 6, 2]]).to (device) 
    
    src_pad_idx = 0 
    trg_pad_idx = 0 
    src_vocab_size = 10 
    trg_vocab_size = 10 
    model = Transformer(src_vocab_size, trg_vocab_size, src_pad_idx, trg_pad_idx).to( 
        device
    )
    out = model(x, trg[:, :-1]) 
    print(out.shape)
    
    print("Custom Transformer Implementation:")
    print(f"Output shape: {out.shape}")
    
    # Get actual predictions
    predictions = torch.argmax(out, dim=-1)
    print("\nPredictions for first sequence:")
    print(f"Input sequence:     {x[0].cpu().numpy()}")
    print(f"Target sequence:    {trg[0].cpu().numpy()}")
    print(f"Predicted sequence: {predictions[0].cpu().numpy()}")

torch.Size([2, 7, 10])
Custom Transformer Implementation:
Output shape: torch.Size([2, 7, 10])

Predictions for first sequence:
Input sequence:     [1 5 6 4 3 9 5 2 0]
Target sequence:    [1 7 4 3 5 9 2 0]
Predicted sequence: [6 0 3 7 9 6 6]


## Hugging Face Implementation for Sequence Translation

In [49]:
# Install required packages
!pip install transformers datasets torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [58]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM, 
    AutoTokenizer, 
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset

# Initialize tokenizer and model (using T5-small for sequence translation)
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use the same data as our custom transformer
x = torch.tensor([[1, 5, 6, 4, 3, 9, 5, 2, 0], [1, 8, 7, 3, 4, 5, 6, 7, 2]])
trg = torch.tensor([[1, 7, 4, 3, 5, 9, 2, 0], [1, 5, 6, 2, 4, 7, 6, 2]])

# Create dataset
dataset = Dataset.from_dict({
    'input_text': [' '.join(map(str, seq.numpy())) for seq in x],
    'target_text': [' '.join(map(str, seq.numpy())) for seq in trg]
})

# Preprocessing function
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples['target_text'],
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Preprocess and setup training
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Training arguments - using Seq2SeqTrainingArguments instead of TrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=1,
    save_steps=100,
    predict_with_generate=True,  # Enable generation during prediction
    generation_max_length=128,    # Set maximum generation length
    generation_num_beams=4        # Set number of beams for generation
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Train the model
# trainer.train()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [59]:
# Move model to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def translate_sequence(input_sequence, max_length=128):
    # Convert input sequence to string format
    input_text = ' '.join(map(str, input_sequence))
    
    # Tokenize input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(device)
    
    # Generate output sequence
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    
    # Decode output tokens to text
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Convert back to number sequence
    return [int(x) for x in decoded.split() if x.isdigit()]

# Test both implementations with same input
x_test = [1, 5, 6, 4, 3, 9, 5, 2, 0]
target = [1, 7, 4, 3, 5, 9, 2, 0]

# Test Hugging Face implementation
output = translate_sequence(x_test)
print("Hugging Face Implementation:")
print(f"Input sequence:  {x_test}")
print(f"Target sequence: {target}")
print(f"Model output:    {output}")

Hugging Face Implementation:
Input sequence:  [1, 5, 6, 4, 3, 9, 5, 2, 0]
Target sequence: [1, 7, 4, 3, 5, 9, 2, 0]
Model output:    [1, 5, 5, 6, 4, 3, 3, 9, 5, 2, 0]
